In [1]:
# ================================================================
#  LCCN-R22b - LEAVE-ONE-OUT ABLATION: XLM-RoBERTa, SEED 42  [GPU, ~6 h]
#  Trains XLM-RoBERTa (trainer v1.4, BUDGET unchanged) on the seven
#  leave-one-out arms written by LCCN-R22a: loo_full and the six
#  minus-one variants. lccn_base at seed 42 is taken from the NB4 output.
#  Per-seed prediction files are also written as __seed42.csv.
#  Add Input: lccn-r22a-loo-arms (output: loo_texts.csv),
#             paper1-nb1-lccn-final (output: splits/),
#             paper1-nb2-lccn-ml (output: trainer.py).
#  Accelerator GPU T4 x2 or P100. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys, os, shutil, glob
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')
for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True); sys.modules.pop("trainer", None)
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "trainer.py not found - attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
import torch, transformers
print("\npython", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU is off - Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.



python 3.12.13 | torch 2.10.0+cu128 | transformers 5.18.0 | cuda True
gpu: Tesla T4


In [2]:
# ================================================================
#  CELL 1 - SHARED SETUP (trainer v1.4 asserted) AND LOO ARM JOIN
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np
FP = trainer.fingerprint()
assert trainer.TRAINER_VERSION == "1.4" and FP == "5b607248090bd26e", FP

sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert sd, "attach the paper1-nb1-lccn-final output"
tr, va, te, HAVE = load_splits(sd[0])

lt = glob.glob("/kaggle/input/**/loo_texts.csv", recursive=True)
assert lt, "attach the lccn-r22a-loo-arms output (loo_texts.csv)"
loo = pd.read_csv(lt[0], dtype=str, keep_default_na=False)
loo.columns = [c.strip().lower() for c in loo.columns]
idc = next(c for c in loo.columns if c in ("id", "comment_id"))
LOO_ARMS = ["loo_full", "loo_no_emotion", "loo_no_negation", "loo_no_length",
            "loo_no_ortho", "loo_no_gtest", "loo_no_cluster"]
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for a in LOO_ARMS:
        f[f"text_{a}"] = ids.map(dict(zip(loo[idc].astype(str), loo[f"text_{a}"])))
        assert f[f"text_{a}"].notna().all(), f"id join failed for {a}"
print("arms joined:", LOO_ARMS)


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arms joined: ['loo_full', 'loo_no_emotion', 'loo_no_negation', 'loo_no_length', 'loo_no_ortho', 'loo_no_gtest', 'loo_no_cluster']


In [3]:
# ================================================================
#  CELL 2 - XLM-RoBERTa, SEED 42, SEVEN ARMS (resumable across sessions)
# ================================================================
MODEL = "XLM-RoBERTa"; TAG = "lccn_r22_loo_xlmr"
res = run_grid(tr, va, te, LOO_ARMS, [MODEL], tag=TAG, seeds_tf=(42,), resume=True)
print("\n=== XLM-RoBERTa seed 42, macro-F1 ===")
print(res.to_string(index=False))

# per-seed copies so the per-seed statistics notebook can identify them
for a in LOO_ARMS:
    src = f"/kaggle/working/preds/{MODEL}__{a}.csv"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/preds/{MODEL}__{a}__seed42.csv")

r = res.iloc[0]
ref = float(r["loo_full"])
print(f"\n{'arm':<18}{'macro-F1':>10}{'vs loo_full':>14}")
for a in LOO_ARMS:
    print(f"{a:<18}{float(r[a]):>10.4f}{float(r[a])-ref:>+14.4f}")
print("\nlccn_base seed 42 (0.7991) and lccn seed 42 (0.8051) from NB4 are the reference rows;")
print("loo_full reproduces lccn up to the one order-dependent decision noted in Section 4.2.")
print("\nDONE - results_lccn_r22_loo_xlmr.csv and preds/ written.")
print("Save & Run All (Commit); transcribe only from the committed log.")


resume: 40 prediction files already on disk


=== XLM-RoBERTa / loo_full ===


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7498
    epoch  2/10  val macro-F1 = 0.7885
    epoch  3/10  val macro-F1 = 0.8003
    epoch  4/10  val macro-F1 = 0.8025
    epoch  5/10  val macro-F1 = 0.8145
    epoch  6/10  val macro-F1 = 0.8070
    epoch  7/10  val macro-F1 = 0.8004
    epoch  8/10  val macro-F1 = 0.8096
    early stop at epoch 8
  -> macro-F1 0.8011

=== XLM-RoBERTa / loo_no_cluster ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7530
    epoch  2/10  val macro-F1 = 0.7799
    epoch  3/10  val macro-F1 = 0.7969
    epoch  4/10  val macro-F1 = 0.8025
    epoch  5/10  val macro-F1 = 0.8095
    epoch  6/10  val macro-F1 = 0.8146
    epoch  7/10  val macro-F1 = 0.8174
    epoch  8/10  val macro-F1 = 0.8064
    epoch  9/10  val macro-F1 = 0.8173
    epoch 10/10  val macro-F1 = 0.8127
    early stop at epoch 10
  -> macro-F1 0.8004

wrote /kaggle/working/results_lccn_r22_loo_xlmr.csv

=== XLM-RoBERTa seed 42, macro-F1 ===
      model      family  loo_full  loo_no_emotion  loo_no_negation  loo_no_length  loo_no_ortho  loo_no_gtest  loo_no_cluster
XLM-RoBERTa transformer    0.8011          0.8061            0.806         0.8023        0.8016        0.8011          0.8004

arm                 macro-F1   vs loo_full
loo_full              0.8011       +0.0000
loo_no_emotion        0.8061       +0.0050
loo_no_negation       0.8060       +0.0049
loo_no_length         0.8023       +0.0012
lo